# ECHR Case Scraper — HUDOC

Downloads ECHR judgments from HUDOC matching research keywords about
parental alienation and child welfare, using the `echr_extractor` library.
Extracts keyword-in-context (KWIC) windows for discourse analysis.

**Dependency:** `pip install echr-extractor`

Master's thesis — notebook version of `echr_scraper.py`.

In [3]:
import json
from collections import Counter

# Load ECHR data
with open("../data/echr/echr_cases.json", "r", encoding="utf-8") as f:
    echr_cases = json.load(f)

print(f"Total ECHR cases: {len(echr_cases)}")

# Country distribution
countries = Counter(c.get("respondent", "?") for c in echr_cases)
print(f"\nCountries represented: {len(countries)}")
for country, n in countries.most_common():
    print(f"  {country}: {n} cases")

# Article violations
violations = Counter()
for c in echr_cases:
    for v in c.get("violation", "").split(";"):
        v = v.strip()
        if v:
            violations[v] += 1
print(f"\nTop violations:")
for v, n in violations.most_common(10):
    print(f"  Article {v}: {n}")

# Keywords
kw_counts = Counter()
for c in echr_cases:
    for kw in c.get("matched_keywords", []):
        kw_counts[kw] += 1
print(f"\nKeywords:")
for kw, n in kw_counts.most_common():
    print(f"  {kw}: {n}")

Total ECHR cases: 117

Countries represented: 34
  NOR: 18 cases
  POL: 8 cases
  HUN: 7 cases
  ROU: 7 cases
  RUS: 7 cases
  GRC: 6 cases
  CZE: 6 cases
  SRB: 5 cases
  CHE: 5 cases
  BGR: 4 cases
  SWE: 4 cases
  SVK: 4 cases
  FIN: 3 cases
  HRV: 3 cases
  MKD: 3 cases
  PRT: 3 cases
  GEO: 2 cases
  LVA: 2 cases
  TUR: 2 cases
  ESP: 2 cases
  MLT: 2 cases
  ITA: 2 cases
  SVN: 1 cases
  GBR: 1 cases
  NLD: 1 cases
  UKR: 1 cases
  MDA: 1 cases
  ALB: 1 cases
  ARM: 1 cases
  CYP: 1 cases
  AUT: 1 cases
  ISL: 1 cases
  LTU: 1 cases
  AZE: 1 cases

Top violations:
  Article 8: 35
  Article 8-1: 34
  Article 13: 7
  Article 3: 6
  Article 14: 6
  Article 6: 5
  Article 6-1: 5
  Article 5: 4
  Article 5-1: 3
  Article 13+3: 3

Keywords:
  contact rights: 69
  best interests of the child: 67
  child welfare: 26
  custody AND alienation: 12
  parental alienation: 4


In [1]:
import json
import logging
import os
import time
from collections import Counter
from datetime import datetime

import echr_extractor as echr

/Users/maksimsmirnov/Desktop/thesis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Configuration

Adjust date range, output path, and per-query limits here.

In [2]:
START_DATE       = "2024-01-01"   # PoC range; extend to "1959-01-01" for full run
END_DATE         = "2025-12-31"

REQUEST_DELAY    = 2.0            # seconds between keyword queries
COUNT_PER_QUERY  = 5000           # max cases per keyword query

OUTPUT_DIR       = "../data/echr"
OUTPUT_CASES     = os.path.join(OUTPUT_DIR, "echr_cases.json")
OUTPUT_KWIC      = os.path.join(OUTPUT_DIR, "echr_kwic.json")
CHECKPOINT_FILE  = os.path.join(OUTPUT_DIR, "echr_checkpoint.json")
CHECKPOINT_EVERY = 50             # checkpoint after this many new cases

## Keywords

Each keyword triggers a separate HUDOC query with **no article filter** —
which articles appear is itself an analytical finding.

In [3]:
KEYWORDS = [
    "parental alienation",
    "child welfare",
    "best interests of the child",
    "contact rights",
    "custody AND alienation",
    "Kindeswohl",
    "Entfremdung",
]

# HUDOC fulltext query string for each keyword label
FULLTEXT_QUERIES = {
    "parental alienation":          '"parental alienation"',
    "child welfare":                '"child welfare"',
    "best interests of the child":  '"best interests of the child"',
    "contact rights":               '"contact rights"',
    "custody AND alienation":       "custody alienation",   # both words anywhere
    "Kindeswohl":                   '"Kindeswohl"',
    "Entfremdung":                  '"Entfremdung"',
}

## Logging Setup

In [4]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

## Helper Functions

In [5]:
def extract_kwic(text, keyword, window=200):
    """Extract all occurrences of keyword with surrounding context (±window chars)."""
    contexts = []
    if not text:
        return contexts
    # "custody AND alienation" is a compound label; search for "alienation"
    search_term = "alienation" if keyword == "custody AND alienation" else keyword
    text_lower = text.lower()
    kw_lower = search_term.lower()
    start = 0
    while True:
        idx = text_lower.find(kw_lower, start)
        if idx == -1:
            break
        ctx_start = max(0, idx - window)
        ctx_end = min(len(text), idx + len(search_term) + window)
        contexts.append({
            "keyword":  keyword,
            "position": idx,
            "context":  text[ctx_start:ctx_end].strip(),
        })
        start = idx + 1
    return contexts


def parse_year(date_str):
    """Parse year from ECHR dates like '19/12/2024 00:00:00' or ISO format."""
    if not date_str:
        return None
    try:
        s = str(date_str).strip()
        if "/" in s:
            return int(s.split("/")[2].split(" ")[0])
        else:
            return datetime.fromisoformat(s[:10]).year
    except Exception:
        return None

In [6]:
def build_hudoc_link(fulltext, start_date, end_date):
    """Build a HUDOC fragment URL for echr_extractor's link_to_query()."""
    params = {
        "documentcollectionid2": ["JUDGMENTS", "COMMUNICATEDCASES", "DECISIONS"],
        "languageisocode": ["ENG"],
        "fulltext": [fulltext],
        "kpdate": [start_date, end_date],
    }
    fragment = json.dumps(params, ensure_ascii=False)
    return f"https://hudoc.echr.coe.int/eng#{fragment}"


def df_to_records(df):
    """Convert a pandas DataFrame to a list of plain dicts (JSON-safe)."""
    if df is False or df is None or len(df) == 0:
        return []
    records = df.to_dict(orient="records")
    for r in records:
        for k, v in r.items():
            if v != v:  # NaN → None
                r[k] = None
    return records

In [7]:
def load_checkpoint():
    """Return (cases_by_id, kwic_list) from checkpoint, or empty defaults."""
    if not os.path.exists(CHECKPOINT_FILE):
        return {}, []
    try:
        with open(CHECKPOINT_FILE, encoding="utf-8") as f:
            data = json.load(f)
        cases = {c["itemid"]: c for c in data.get("cases", []) if c.get("itemid")}
        kwic  = data.get("kwic", [])
        log.info(f"Checkpoint loaded: {len(cases)} cases, {len(kwic)} KWIC entries")
        return cases, kwic
    except Exception as e:
        log.warning(f"Could not load checkpoint ({e}) — starting fresh")
        return {}, []


def save_checkpoint(cases_by_id, kwic_list):
    """Write current state to checkpoint file."""
    try:
        data = {
            "saved_at": datetime.now().isoformat(timespec="seconds"),
            "cases":    list(cases_by_id.values()),
            "kwic":     kwic_list,
        }
        with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        log.info(f"Checkpoint saved: {len(cases_by_id)} cases, {len(kwic_list)} KWIC entries")
    except Exception as e:
        log.warning(f"Could not save checkpoint: {e}")

In [8]:
def run_keyword_query(keyword):
    """
    Fetch cases for one keyword from HUDOC.
    Tries get_echr_extra first (returns full texts), falls back to get_echr on failure.
    Returns (records, full_texts_dict) where full_texts_dict maps itemid → text.
    """
    fulltext_q = FULLTEXT_QUERIES[keyword]
    link = build_hudoc_link(fulltext_q, START_DATE, END_DATE)
    log.info(f"[{keyword}] Querying HUDOC …")

    try:
        df, full_texts = echr.get_echr_extra(
            link=link,
            count=COUNT_PER_QUERY,
            language=["ENG"],
            save_file="n",
            verbose=False,
            progress_bar=True,
        )
        records = df_to_records(df)
        # full_texts is a list of {"item_id": ..., "ecli": ..., "full_text": ...}
        # normalise to {itemid: text} dict for uniform downstream handling
        ft = {}
        if isinstance(full_texts, list):
            for item in full_texts:
                iid = item.get("item_id")
                text = item.get("full_text", "")
                if iid and text:
                    ft[iid] = text
        elif isinstance(full_texts, dict):
            ft = full_texts
        log.info(f"[{keyword}] → {len(records)} cases (full texts: {len(ft)})")
        return records, ft
    except Exception as exc:
        log.warning(f"[{keyword}] get_echr_extra failed ({exc}), falling back to get_echr")

    try:
        df = echr.get_echr(
            link=link,
            count=COUNT_PER_QUERY,
            language=["ENG"],
            save_file="n",
            verbose=False,
            progress_bar=True,
        )
        records = df_to_records(df)
        log.info(f"[{keyword}] → {len(records)} cases (metadata only)")
        return records, {}
    except Exception as exc2:
        log.error(f"[{keyword}] Both queries failed: {exc2}")
        return [], {}

## Phase 1 — Collect Cases

Run all keyword queries and deduplicate. Cases matching multiple keywords get a
`matched_keywords` list. Resumes from checkpoint if one exists.

In [9]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Resume from checkpoint if available
cases_by_id, kwic_list = load_checkpoint()
log.info(f"Starting with {len(cases_by_id)} cases already in checkpoint")

# Pre-populate full texts from checkpoint
full_texts_by_id = {
    cid: c.get("full_text", "") for cid, c in cases_by_id.items()
}
keyword_raw_counts = {}

INFO:__main__:Starting with 0 cases already in checkpoint


In [10]:
for i, keyword in enumerate(KEYWORDS):
    records, full_texts = run_keyword_query(keyword)
    keyword_raw_counts[keyword] = len(records)

    for rec in records:
        item_id = rec.get("itemid") or rec.get("DocId") or rec.get("WorkId")
        if item_id is None:
            continue
        if item_id not in cases_by_id:
            cases_by_id[item_id] = rec
            cases_by_id[item_id]["matched_keywords"] = []
        if keyword not in cases_by_id[item_id].get("matched_keywords", []):
            cases_by_id[item_id].setdefault("matched_keywords", []).append(keyword)

    for item_id, text in full_texts.items():
        if text:
            full_texts_by_id[item_id] = text

    if i < len(KEYWORDS) - 1:
        log.info(f"Sleeping {REQUEST_DELAY}s …")
        time.sleep(REQUEST_DELAY)

print(f"Unique cases after all queries: {len(cases_by_id)}")

INFO:__main__:[parental alienation] Querying HUDOC …
INFO:root:
--- DONE ---
INFO:root:Full-text download will now begin
INFO:root:Full-text download finished
INFO:__main__:[parental alienation] → 4 cases (full texts: 4)
INFO:__main__:Sleeping 2.0s …
INFO:__main__:[child welfare] Querying HUDOC …
INFO:root:
--- DONE ---
INFO:root:Full-text download will now begin
INFO:root:Full-text download finished
INFO:__main__:[child welfare] → 26 cases (full texts: 26)
INFO:__main__:Sleeping 2.0s …
INFO:__main__:[best interests of the child] Querying HUDOC …
INFO:root:
--- DONE ---
INFO:root:Full-text download will now begin
INFO:root:Full-text download finished
INFO:__main__:[best interests of the child] → 67 cases (full texts: 67)
INFO:__main__:Sleeping 2.0s …
INFO:__main__:[contact rights] Querying HUDOC …
INFO:root:
--- DONE ---
INFO:root:Full-text download will now begin
INFO:root:Full-text download finished
INFO:__main__:[contact rights] → 69 cases (full texts: 69)
INFO:__main__:Sleeping 2.0

Unique cases after all queries: 117


In [11]:
for item_id, case in cases_by_id.items():
    text = full_texts_by_id.get(item_id) or ""
    case["full_text"]   = text
    case["text_length"] = len(text)

n_with = sum(1 for c in cases_by_id.values() if c.get("full_text"))
print(f"Cases with full text: {n_with} / {len(cases_by_id)}")

Cases with full text: 117 / 117


## Phase 2 — KWIC Extraction

For each case's full text, find every occurrence of every matched keyword and
extract a ±200-character context window.

In [12]:
kwic_done_ids = {entry["itemid"] for entry in kwic_list}
new_since_checkpoint = 0

for item_id, case in cases_by_id.items():
    if item_id in kwic_done_ids:
        continue

    full_text  = case.get("full_text") or ""
    doc_name   = case.get("docname", "")
    respondent = case.get("respondent", "")
    year       = parse_year(
        case.get("judgementdate") or case.get("referencedate") or ""
    )

    for keyword in case.get("matched_keywords", []):
        for hit in extract_kwic(full_text, keyword):
            kwic_list.append({
                "itemid":     item_id,
                "docname":    doc_name,
                "year":       year,
                "respondent": respondent,
                "keyword":    hit["keyword"],
                "position":   hit["position"],
                "context":    hit["context"],
            })

    kwic_done_ids.add(item_id)
    new_since_checkpoint += 1

    if new_since_checkpoint >= CHECKPOINT_EVERY:
        save_checkpoint(cases_by_id, kwic_list)
        new_since_checkpoint = 0

save_checkpoint(cases_by_id, kwic_list)
print(f"KWIC contexts extracted: {len(kwic_list)}")

INFO:__main__:Checkpoint saved: 117 cases, 425 KWIC entries
INFO:__main__:Checkpoint saved: 117 cases, 767 KWIC entries
INFO:__main__:Checkpoint saved: 117 cases, 797 KWIC entries


KWIC contexts extracted: 797


## Phase 3 — Save Outputs

In [13]:
all_cases = list(cases_by_id.values())

with open(OUTPUT_CASES, "w", encoding="utf-8") as f:
    json.dump(all_cases, f, ensure_ascii=False, indent=2)
print(f"Cases saved  → {OUTPUT_CASES}  ({len(all_cases)} cases)")

with open(OUTPUT_KWIC, "w", encoding="utf-8") as f:
    json.dump(kwic_list, f, ensure_ascii=False, indent=2)
print(f"KWIC saved   → {OUTPUT_KWIC}  ({len(kwic_list)} entries)")

Cases saved  → ../data/echr/echr_cases.json  (117 cases)
KWIC saved   → ../data/echr/echr_kwic.json  (797 entries)


## Summary Statistics

In [14]:
n_total     = len(all_cases)
n_with_text = sum(1 for c in all_cases if c.get("full_text"))

year_counter       = Counter()
respondent_counter = Counter()
for c in all_cases:
    yr = parse_year(c.get("judgementdate") or c.get("referencedate") or "")
    if yr:
        year_counter[yr] += 1
    respondent_counter[c.get("respondent") or "?"] += 1

kwic_per_keyword = Counter(e["keyword"] for e in kwic_list)
kw_case_counts = {
    kw: sum(1 for c in all_cases if kw in (c.get("matched_keywords") or []))
    for kw in KEYWORDS
}

print(f"{'='*60}")
print(f"Total unique cases        : {n_total}")
print(f"Cases with full text      : {n_with_text}")
print(f"Cases without full text   : {n_total - n_with_text}")
print(f"Total KWIC contexts       : {len(kwic_list)}")
print(f"{'-'*60}")
print("Cases per keyword (unique matches):")
for kw in KEYWORDS:
    print(f"  {kw_case_counts.get(kw, 0):>5}  {kw}")
print(f"{'-'*60}")
print("KWIC contexts per keyword:")
for kw in KEYWORDS:
    print(f"  {kwic_per_keyword.get(kw, 0):>5}  {kw}")
print(f"{'-'*60}")
print("Cases per year:")
for yr, cnt in sorted(year_counter.items(), reverse=True):
    print(f"  {yr}: {cnt}")
print(f"{'-'*60}")
print("Top 5 respondent countries:")
for country, cnt in respondent_counter.most_common(5):
    print(f"  {country}: {cnt}")
print(f"{'='*60}")

Total unique cases        : 117
Cases with full text      : 117
Cases without full text   : 0
Total KWIC contexts       : 797
------------------------------------------------------------
Cases per keyword (unique matches):
      4  parental alienation
     26  child welfare
     67  best interests of the child
     69  contact rights
     12  custody AND alienation
      0  Kindeswohl
      0  Entfremdung
------------------------------------------------------------
KWIC contexts per keyword:
      9  parental alienation
    164  child welfare
    225  best interests of the child
    367  contact rights
     32  custody AND alienation
      0  Kindeswohl
      0  Entfremdung
------------------------------------------------------------
Cases per year:
  2025: 26
  2024: 34
------------------------------------------------------------
Top 5 respondent countries:
  NOR: 18
  POL: 8
  HUN: 7
  ROU: 7
  RUS: 7
